In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Vzdálená transpilace obvodů pomocí Qiskit Transpiler Service

> **Danger:** Od 18. července 2025 probíhá migrace služby na podporu nové platformy IBM Quantum&reg; Platform a služba není dostupná. Pro AI průchody můžeš použít [lokální režim](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud).
> 
>     Služba je ve fázi beta vydání a může se změnit.
>     Pokud máš zpětnou vazbu nebo chceš kontaktovat tým vývojářů, použij prosím tento kanál [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Qiskit Transpiler Service poskytuje schopnosti transpilace v cloudu. Kromě lokálních schopností Qiskit Transpileru mohou tvoje transpilační úlohy využívat cloudové prostředky IBM Quantum i AI-podporované průchody Transpileru.

Qiskit Transpiler Service nabízí knihovnu pro Python, která tuto službu a její schopnosti plynule integruje do tvých stávajících vzorců a pracovních postupů s Qiskitem. Tato služba je dostupná pouze pro uživatele plánů IBM Quantum Premium Plan, Flex Plan a On-Prem (přes IBM Quantum Platform API) Plan.

<span id="install-transpiler-service"></span>
## Instalace balíčku qiskit-ibm-transpiler
Chceš-li používat Qiskit Transpiler Service, nainstaluj balíček `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

Balíček se automaticky ověřuje pomocí tvých [přihlašovacích údajů IBM Quantum Platform](/guides/cloud-setup) v souladu se způsobem, jakým [to spravuje Qiskit Runtime](/guides/initialize-account):
- Proměnná prostředí: `QISKIT_IBM_TOKEN`
- Konfigurační soubor `~/.qiskit/qiskit-ibm.json` (v sekci `default-ibm-quantum`).

*Poznámka*: Tento balíček vyžaduje Qiskit SDK v1.X.

## Možnosti transpilace qiskit-ibm-transpiler
- `backend_name` (volitelné, str) – Název Backend, jak by byl očekáván službou QiskitRuntimeService (například `ibm_torino`). Je-li tato možnost nastavena, použije metoda transpile pro transpilaci rozvržení ze zadaného Backend. Pokud je nastavena jiná možnost, která tato nastavení ovlivňuje, například `coupling_map`, jsou nastavení `backend_name` přepsána.
- `coupling_map` (volitelné, List[List[int]]) – Platný seznam coupling map (například [[0,1],[1,2]]). Je-li tato možnost nastavena, použije metoda transpile tuto coupling map pro transpilační operaci. Je-li definována, přepíše jakoukoli hodnotu zadanou pro `target`.
- `optimization_level` (int) – Potenciální úroveň optimalizace, která se použije během transpilačního procesu. Platné hodnoty jsou [1,2,3], kde 1 znamená nejmenší optimalizaci (a nejrychlejší) a 3 nejvyšší optimalizaci (a nejnáročnější na čas).
- `ai` ("true", "false", "auto") – Zda se mají během transpilace použít schopnosti s podporou AI. Dostupné schopnosti s podporou AI mohou zahrnovat průchody `AIRouting` nebo jiné metody syntézy s podporou AI. Je-li tato hodnota `"true"`, použije služba různé průchody transpilace s podporou AI v závislosti na požadované `optimization_level`. Při hodnotě `"false"` se použijí nejnovější funkce transpilace Qiskitu bez AI. A konečně při hodnotě `"auto"` služba sama rozhodne, zda použije standardní heuristické průchody Qiskitu nebo průchody s podporou AI na základě tvého Circuit.
- `qiskit_transpile_options` (dict) – Objekt Python slovníku, který může obsahovat jakoukoli jinou platnou možnost [metody Qiskit `transpile()`](defaults-and-configuration-options). Pokud vstup `qiskit_transpile_options` obsahuje `optimization_level`, je ignorován ve prospěch `optimization_level` zadaného jako parametr. Pokud `qiskit_transpile_options` obsahuje jakoukoli možnost, kterou metoda Qiskit `transpile()` nerozpozná, knihovna vyvolá chybu.

Více informací o dostupných metodách `qiskit-ibm-transpiler` najdeš v [referenci API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). Chceš-li se dozvědět více o API služby, podívej se na [dokumentaci REST API Qiskit Transpiler Service.](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## Příklady
Následující příklady ukazují, jak transpilovat Circuit pomocí Qiskit Transpiler Service s různými parametry.

1. Vytvoř Circuit a zavolej Qiskit Transpiler Service pro transpilaci obvodu s `ibm_torino` jako `backend_name`, hodnotou 3 pro `optimization_level` a bez použití AI během transpilace.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*Poznámka*: Jako backend_name můžeš použít pouze zařízení, ke kterým máš přístup se svým účtem IBM Quantum. Kromě `backend_name` umožňuje `TranspilerService` jako parametr také `coupling_map`.

2. Vytvoř podobný Circuit a transpiluj ho s požadavkem na schopnosti transpilace s AI nastavením příznaku `ai` na `True`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. Vytvoř podobný Circuit a transpiluj ho tak, aby se služba sama rozhodla, zda použít průchody transpilace s podporou AI.